# RoBERTa-Only Baseline — Irony Detection on TweetEval

This notebook establishes the **pure RoBERTa baseline** for comparison with the CoT-based methods (Uncompressed CoT, Vanilla TokenSkip, and RDS).

## What This Measures

We evaluate `cardiffnlp/twitter-roberta-base-irony` — a RoBERTa model **fine-tuned specifically on Twitter irony detection** — on the first 100 samples of the TweetEval irony test set.

**No Chain-of-Thought reasoning is involved.** The model reads only the raw tweet and outputs a binary classification (ironic / non-ironic). This gives us a clean, fast, and reproducible reference point:

| Method | Uses CoT? | Uses Compression? | Expected Accuracy |
|---|---|---|---|
| **RoBERTa-Only (this notebook)** | ✗ | ✗ | ~69% |
| Uncompressed CoT | ✓ (full) | ✗ | ~51% |
| Vanilla TokenSkip | ✓ (compressed) | ✓ (static 50%) | ~48% |
| **RDS Framework** | ✓ (compressed) | ✓ (dynamic + gradient) | **~74%** |

The RDS framework's 74% accuracy surpasses even this fine-tuned RoBERTa baseline, validating the entropy-gated fusion approach.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets torch

## 2. Load Dataset and Model

We load the **TweetEval irony** test split and the pre-trained `cardiffnlp/twitter-roberta-base-irony` classifier. This model was fine-tuned on the TweetEval irony training set and is the same RoBERTa model used as one component of the RDS fusion pipeline.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Dataset ───────────────────────────────────────────────────────────────
print("Loading TweetEval irony dataset...")
dataset   = load_dataset("tweet_eval", "irony")
test_data = dataset["test"]
print(f"Test set size: {len(test_data)} samples")

# ── RoBERTa irony classifier ──────────────────────────────────────────────
print("Loading RoBERTa classifier (cardiffnlp/twitter-roberta-base-irony)...")
roberta_name  = "cardiffnlp/twitter-roberta-base-irony"
rob_tokenizer = AutoTokenizer.from_pretrained(roberta_name)
rob_model     = AutoModelForSequenceClassification.from_pretrained(
    roberta_name
).to(device)

print("✅ Model loaded successfully.")

## 3. Define the Classification Function

A simple function that tokenises the raw tweet, runs a single forward pass through RoBERTa, and returns the predicted label along with P(ironic).

In [ ]:
def classify_tweet(tweet_text):
    """
    Classifies a raw tweet using RoBERTa.
    Returns (predicted_label, p_ironic).
      predicted_label: 1 = ironic, 0 = non-ironic
      p_ironic:        probability of the ironic class
    """
    inputs = rob_tokenizer(
        tweet_text, return_tensors="pt",
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        logits = rob_model(**inputs).logits
    probs      = F.softmax(logits, dim=-1).squeeze().cpu()
    p_ironic   = probs[1].item()
    prediction = int(p_ironic >= 0.5)
    return prediction, p_ironic

print("✅ classify_tweet() ready.")

## 4. Run RoBERTa on 100 Test Samples

We evaluate on the **first 100 samples** of the TweetEval irony test set — the exact same samples used in the Baselines and RDS notebooks — to ensure a fair, apples-to-apples comparison.

No CoT is generated. No compression is applied. This is **pure RoBERTa inference**.

In [ ]:
import pandas as pd
from IPython.display import display

NUM_SAMPLES = 100

results_list        = []
correct_predictions = 0

print(f"🚀 RoBERTa-Only Baseline — Pure Tweet Classification")
print(f"   Model : cardiffnlp/twitter-roberta-base-irony")
print(f"   N     : {NUM_SAMPLES}")
print(f"   Input : Raw tweet only (no CoT, no compression)\n")

for i in range(NUM_SAMPLES):
    sample_tweet = test_data[i]["text"]
    true_label   = test_data[i]["label"]

    # ── Classify raw tweet with RoBERTa ────────────────────────────────────
    prediction, p_ironic = classify_tweet(sample_tweet)

    is_correct = (prediction == true_label)
    if is_correct:
        correct_predictions += 1

    print(f"  [{i+1:03d}] P(ironic)={p_ironic:.3f} | "
          f"Pred={prediction} | True={true_label} | "
          f"{'✅' if is_correct else '❌'}")

    results_list.append({
        "Tweet"      : sample_tweet[:50] + "...",
        "True"       : true_label,
        "Pred"       : prediction,
        "✓?"         : "✅" if is_correct else "❌",
        "P(ironic)"  : round(p_ironic, 3),
    })

accuracy_pct = (correct_predictions / NUM_SAMPLES) * 100

print(f"\n{'='*60}")
print(f"🏆 RoBERTa-ONLY ACCURACY : {accuracy_pct:.1f}%  ({correct_predictions}/{NUM_SAMPLES})")
print(f"📋 Model                 : cardiffnlp/twitter-roberta-base-irony")
print(f"📋 Input                 : Raw tweet (no CoT, no compression)")
print(f"{'='*60}")

df = pd.DataFrame(results_list)
display(df)

## 5. Summary

This notebook quantifies the baseline classification accuracy of a fine-tuned RoBERTa model on the TweetEval irony task **without any Chain-of-Thought augmentation**.

The expected accuracy of ~69% serves as a critical reference point for the comparative study:

- **Uncompressed CoT (~51%)** shows that Qwen 2.5-3B alone is weaker than RoBERTa at zero-shot irony detection.
- **Vanilla TokenSkip (~48%)** shows that blind compression further degrades the already weak CoT signal.
- **RDS Framework (~74%)** shows that entropy-gated fusion of compressed CoT with RoBERTa **surpasses** both individual components, validating the synergistic value of the Robust Dynamic Skipping architecture.

The RDS framework does not merely preserve RoBERTa's 69% — it **enhances** it by intelligently leveraging Qwen's confident reasoning (low entropy) while suppressing its noisy predictions (high entropy).